In [125]:
# list of countries generated by gemini

countries_label = {
    "Albania": 0,
    "Andorra": 1,
    "Austria": 2,
    "Belarus": 3,
    "Belgium": 4,
    "Bosnia and Herzegovina": 5,
    "Bulgaria": 6,
    "Croatia": 7,
    "Czech Republic": 8,
    "Denmark": 9,
    "Estonia": 10,
    "Finland": 11,
    "France": 12,
    "Germany": 13,
    "Greece": 14,
    "Hungary": 15,
    "Iceland": 16,
    "Ireland": 17,
    "Italy": 18,
    "Kosovo": 19,
    "Latvia": 20,
    "Liechtenstein": 21,
    "Lithuania": 22,
    "Luxembourg": 23,
    "Malta": 24,
    "Moldova": 25,
    "Monaco": 26,
    "Montenegro": 27,
    "Netherlands": 28,
    "North Macedonia": 29,
    "Norway": 30,
    "Poland": 31,
    "Portugal": 32,
    "Romania": 33,
    "San Marino": 34,
    "Serbia": 35,
    "Slovakia": 36,
    "Slovenia": 37,
    "Spain": 38,
    "Sweden": 39,
    "Switzerland": 40,
    "Turkey": 41,
    "Ukraine": 42,
    "United Kingdom": 43,
    "Vatican City": 44
}

countries = {
    0: [14, 19, 27, 29],
    1: [12, 38],
    2: [8, 13, 15, 18, 21, 36, 37, 40],
    3: [20, 22, 31, 42],
    4: [12, 13, 23, 28],
    5: [7, 27, 35],
    6: [14, 29, 33, 35, 41],
    7: [5, 15, 27, 35, 37],
    8: [2, 13, 31, 36],
    9: [13],
    10: [20],
    11: [30, 39],
    12: [1, 4, 13, 18, 23, 26, 38, 40],
    13: [2, 4, 8, 9, 12, 23, 28, 31, 40],
    14: [0, 6, 29, 41],
    15: [2, 7, 33, 35, 36, 37, 42],
    16: [],
    17: [43],
    18: [2, 12, 34, 40, 44],
    19: [0, 27, 29, 35],
    20: [3, 10, 22],
    21: [2, 40],
    22: [3, 20, 31],
    23: [4, 12, 13],
    24: [],
    25: [33, 42],
    26: [12],
    27: [0, 5, 7, 19, 35],
    28: [4, 13],
    29: [0, 6, 14, 19, 35],
    30: [11, 39],
    31: [3, 8, 13, 22, 36, 42],
    32: [38],
    33: [6, 15, 25, 35, 42],
    34: [18],
    35: [5, 6, 7, 15, 19, 27, 29, 33],
    36: [2, 8, 15, 31, 42],
    37: [2, 7, 15, 18],
    38: [1, 12, 32],
    39: [11, 30],
    40: [2, 12, 13, 18, 21],
    41: [6, 14],
    42: [3, 15, 25, 31, 33, 36],
    43: [17],
    44: [18]
}


In [123]:
inverse_map = {v: k for k, v in countries_label.items()}
numbers = [3, 10, 11, 20, 22, 30, 31, 43]

country_names = [inverse_map[num] for num in numbers]


In [124]:
country_names

['Belarus',
 'Estonia',
 'Finland',
 'Latvia',
 'Lithuania',
 'Norway',
 'Poland',
 'Ukraine']

In [ ]:
import random 
import numpy

class Individual:
    def __init__(self, value: str, age: int):
        self.value = value
        self.age = age
        self.score = self.fitness(value)

    def fitness(self, value: str):
        score = 0
        for i in range(len(value)):
            for j in countries[i]:
                if value[i] == value[j] and j > i:
                    score += 1
        return score

class Population:
    def __init__(self, n_pop: int, max_len: int, colors: int):
        self.n_pop = n_pop
        self.max_len = max_len
        self.individuals = []
        self.colors = colors
        self.labels = ''.join(str(i) for i in range(colors))


    def populate(self):
        self.individuals = [Individual(self.random_individual(), 0) for _ in range(self.n_pop)]


    def random_individual(self):
        return ''.join(random.choice(self.labels) for _ in range(self.max_len))
    

    def fitness(self, individual: Individual):
        score = 0
        value = individual.value
        for i in range(len(value)):
            for j in countries[i]:
                if value[i] == value[j] and j > i:
                    score += 1
        return score
    

    def fitness_scores(self, parents):
        fitness_scores = []
        for parent in parents:
            fitness_scores += [self.fitness(parent)]
        return fitness_scores

    
    def tournament(self, parents, fitness_scores):
        best_parent = parents[0]
        for i in range(1, len(parents)):
            if parents[i].value < best_parent.value: # looks for the lowest value, since we want to reach 0
                best_parent = parents[i]
        return best_parent

        
    def inverse_transformation(self, score):
        return 1/(score + 1)
    

    def parent_selection(self, parents, fitness_scores, selection_type, n = 7):
        if selection_type == 'RWS':
            probabilites = []
            chosen_parents = []

            total_scores = sum([self.inverse_transformation(score) for score in fitness_scores])

            for score in fitness_scores:
                f_ = self.inverse_transformation(score)
                probabilites += [f_/total_scores]

            while len(chosen_parents) != 2:
                r = random.random()
                for i in range(len(probabilites)): # iterate through probabilites. If random chance is within range of a given parent's slice of wheel, they are selected
                    r -= probabilites[i] 

                    if (i == len(probabilites) - 1) or (r <= 0):
                        if parents[i] not in chosen_parents:
                            chosen_parents.append(parents[i])
                            break
        
        elif selection_type == 'Tournament':
            chosen_parents = []

            while len(chosen_parents) < 2:
                selected_parents = random.sample(parents, n)
                
                candidate = self.tournament(selected_parents)

                if candidate not in chosen_parents:
                    chosen_parents.append(candidate)

        return chosen_parents[0], chosen_parents[1]


    def crossover(self, parents, crossover:str, mutation_type: str,  mutation_rate=0.1):
        new_generation = Population(self.n_pop, self.max_len, self.colors)
        fitness_scores = self.fitness_scores(parents)
        
        if crossover == 'Two point':
            for _ in range(self.n_pop//2):
                p1, p2 = self.rws_parent_selection(parents, fitness_scores)

                p1_value, p2_value = p1.value, p2.value

                cutoff1 = random.randint(1, self.max_len - 2)
                cutoff2 = random.randint(cutoff1 + 1, self.max_len - 1)

                c1_value = p1_value[:cutoff1] + p2_value[cutoff1:cutoff2] + p1_value[cutoff2:]
                c2_value = p2_value[:cutoff1] + p1_value[cutoff1:cutoff2] + p2_value[cutoff2:]

                c1 = Individual(c1_value, 0)
                c2 = Individual(c2_value, 0)

                if random.random() < mutation_rate:
                    c1 = self.mutation(c1, mutation_type)
                    
                if random.random() < mutation_rate:
                    c2 = self.mutation(c2, mutation_type)            
            
                new_generation.individuals.append(c1)
                new_generation.individuals.append(c2)

        elif crossover == 'Uniform':
            a = 2

        return new_generation
    

    def mutation(self, individual: Individual, mutation: str):
        if mutation == 'Swap':
            value = individual.value

            swap1 = random.randint(0, self.max_len)
            swap2 = random.randint(0, self.max_len)

            temp = value[swap1]

            value = value[:swap1] + value[swap2] + value[swap1 + 1:]
            value = value[:swap2] + temp + value[swap2 + 1:]

            individual.value = value

        elif mutation == 'Inversion':
            value = individual.value

            cutoff1 = random.randint(1, self.max_len - 2)
            cutoff2 = random.randint(cutoff1 + 1, self.max_len - 1)

            inversion = value[cutoff1:cutoff2]

            new_value = value[:cutoff1] + inversion[::-1] + value[cutoff2:]
            individual.value = new_value
        
        return individual
    

    def survivor_selection(self, parents, new_generation, selection_type: str):
        if selection_type == 'Elitism':
            total_individuals = parents + new_generation

            fitness_scores = [self.fitness(individual) for individual in total_individuals]

            sorted_arrays = sorted(zip(total_individuals, fitness_scores), key=lambda x: x[1])

            sliced_sorted_arrays = sorted_arrays[:self.n_pop] # gets the n_pop best individuals

            survivor_generation, _ = zip(*sliced_sorted_arrays)
        
        elif selection_type == 'Age based':
            age_sorted_parents = sorted(parents, key=lambda x : (x.age, x.value), reverse=True)
            surviving_parents = age_sorted_parents[k:]

            best_children = sorted(new_generation, key=lambda x: (x.value, x.age))[:k]

            survivor_generation = surviving_parents + best_children

        return list(survivor_generation)

IndentationError: expected an indented block after 'elif' statement on line 163 (3682061223.py, line 166)

In [138]:
population = Population(200, len(countries), 4)

population.populate()

parents = population.individuals

for i in range(20000):
    print('iteration:', i)
    new_generation = population.two_point_cross_over(parents)
    survivor_selection = population.elitism_survivor_selection(parents, new_generation.individuals)

    best_value = sorted(survivor_selection, key = lambda x: x.score)[0]

    print(f'best value: {best_value.value}, best fitness: {best_value.score}')

    parents = survivor_selection

    if best_value.score == 0:
        break

  



iteration: 0
best value: 013213111112022101220213121213103200200031313, best fitness: 8
iteration: 1
best value: 013213111112022101220213121213103200200031313, best fitness: 8
iteration: 2
best value: 013213111112022101220213121213103200200031313, best fitness: 8
iteration: 3
best value: 013213111112022101220213121213103200200031313, best fitness: 8
iteration: 4
best value: 013213111112022101220213121213103200200031313, best fitness: 8
iteration: 5
best value: 013213111112022230220213121213103200201113313, best fitness: 7
iteration: 6
best value: 013213111112022230220213121213103200200013313, best fitness: 6
iteration: 7
best value: 013213111112022230220213121213103200200013313, best fitness: 6
iteration: 8
best value: 013213111012022230220213123200130033013213113, best fitness: 5
iteration: 9
best value: 013213111012022230220213123200130033013213113, best fitness: 5
iteration: 10
best value: 013213111012022230220213123200130033013213113, best fitness: 5
iteration: 11
best value: 01321

In [119]:
for i in population.individuals:
    print(population.fitness(i))

17
15
28
22
19
23
25
17
18
19


In [128]:
for i in survivor_selection:
    print(population.fitness(i))

3
3
3
3
3
3
3
3
3
3


In [139]:
# The string you provided
solution_string = "113212111113022230220213121333131310003013111"

# Your color palette mapping
# (Assuming 0=Red, 1=Green, 2=Blue, 3=Yellow - adjust as needed)
color_map = {
    '0': "Red",
    '1': "Green",
    '2': "Blue",
    '3': "Yellow"
}

# The tabbed labels dictionary we just built
countries_label = {
    "Albania": 0, "Andorra": 1, "Austria": 2, "Belarus": 3, "Belgium": 4,
    "Bosnia and Herzegovina": 5, "Bulgaria": 6, "Croatia": 7, "Czech Republic": 8,
    "Denmark": 9, "Estonia": 10, "Finland": 11, "France": 12, "Germany": 13,
    "Greece": 14, "Hungary": 15, "Iceland": 16, "Ireland": 17, "Italy": 18,
    "Kosovo": 19, "Latvia": 20, "Liechtenstein": 21, "Lithuania": 22,
    "Luxembourg": 23, "Malta": 24, "Moldova": 25, "Monaco": 26, "Montenegro": 27,
    "Netherlands": 28, "North Macedonia": 29, "Norway": 30, "Poland": 31,
    "Portugal": 32, "Romania": 33, "San Marino": 34, "Serbia": 35,
    "Slovakia": 36, "Slovenia": 37, "Spain": 38, "Sweden": 39, "Switzerland": 40,
    "Turkey": 41, "Ukraine": 42, "United Kingdom": 43, "Vatican City": 44
}

def print_map_coloring(labels, chromosome, palette):
    # Sort labels by index to match the string positions
    sorted_countries = sorted(labels.items(), key=lambda x: x[1])
    
    print(f"{'COUNTRY':<25} | {'COLOR'}")
    print("-" * 35)
    
    for name, index in sorted_countries:
        # Get the color digit from the string at the current country's index
        color_digit = chromosome[index]
        # Map the digit to a human-readable color
        color_name = palette.get(color_digit, "Unknown")
        
        print(f"{name:<25} | {color_name}")

# Execute
print_map_coloring(countries_label, solution_string, color_map)

COUNTRY                   | COLOR
-----------------------------------
Albania                   | Green
Andorra                   | Green
Austria                   | Yellow
Belarus                   | Blue
Belgium                   | Green
Bosnia and Herzegovina    | Blue
Bulgaria                  | Green
Croatia                   | Green
Czech Republic            | Green
Denmark                   | Green
Estonia                   | Green
Finland                   | Yellow
France                    | Red
Germany                   | Blue
Greece                    | Blue
Hungary                   | Blue
Iceland                   | Yellow
Ireland                   | Red
Italy                     | Blue
Kosovo                    | Blue
Latvia                    | Red
Liechtenstein             | Blue
Lithuania                 | Green
Luxembourg                | Yellow
Malta                     | Green
Moldova                   | Blue
Monaco                    | Green
Montenegro             

In [61]:
population.fitness(population.individuals[0])

0 14
0 19
0 27
0 29
1 12
1 39
2 8
2 13
2 15
2 18
2 21
2 37
2 38
2 41
3 20
3 22
3 31
3 34
3 43
4 12
4 13
4 23
4 28
5 7
5 27
5 36
6 14
6 29
6 33
6 36
6 42
7 5
7 15
7 27
7 36
7 38
8 2
8 13
8 31
8 37
9 13
10 20
10 34
11 30
11 34
11 40
12 1
12 4
12 13
12 18
12 23
12 26
12 39
12 41
13 2
13 4
13 8
13 9
13 12
13 23
13 28
13 31
13 41
14 0
14 6
14 29
14 42
15 2
15 7
15 33
15 36
15 37
15 38
15 43
17 44
18 2
18 12
18 35
18 41
18 45
19 0
19 27
19 29
19 36
20 3
20 10
20 22
20 34
21 2
21 41
22 3
22 20
22 31
22 34
23 4
23 12
23 13
25 33
25 43
26 12
27 0
27 5
27 7
27 19
27 36
28 4
28 13
29 0
29 6
29 14
29 19
29 36
30 11
30 34
30 40
31 3
31 8
31 13
31 22
31 34
31 37
31 43
32 39
33 6
33 15
33 25
33 36
33 43
34 3
34 10
34 11
34 20
34 22
34 30
34 31
34 43
35 18
36 5
36 6
36 7
36 15
36 19
36 27
36 29
36 33
37 2
37 8
37 15
37 31
37 43
38 2
38 7
38 15
38 18
39 1
39 12
39 32
40 11
40 30
41 2
41 12
41 13
41 18
41 21
42 6
42 14
43 3
43 15
43 25
43 31
43 33
43 34
43 37
44 17
45 18


50